# Notebook 03 v2 — UNSW-NB15 calibration (hybrid Platt/isotonic)

**What this notebook does:**

Fits per-class calibrators on UNSW calibration set, applies to test set, reports per-class Brier and ECE. Mirrors the NSL and CIC hybrid notebooks. Completes the three-dataset hybrid matrix.

**UNSW calibration set sizes:**
- Normal: 11,200 → isotonic
- DoS: 2,453 → isotonic
- Probe: 2,498 → isotonic
- R2L: 10,665 → isotonic
- U2R: 253 → isotonic (above n=30 threshold, so all classes get isotonic on UNSW)

**Note:** UNSW is the only dataset where every class has n_calib ≥ 30, so the hybrid strategy reduces to pure isotonic. The 'hybrid' label is still applied for consistency with NSL and CIC documentation.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'unsw_nb15_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PROBS_DIR = MODELS_DIR / 'probabilities'   # UNSW uses split predictions/ and probabilities/
CALIB_OUT_DIR = Path(REPO) / 'calibrators' / DATASET
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

PLATT_THRESHOLD = 30
ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Probabilities source: {PROBS_DIR}')
print(f'Output dir: {CALIB_OUT_DIR}')

Dataset: unsw_nb15_v2
Probabilities source: /content/drive/MyDrive/XIDS_Research/xids-research/models/unsw_nb15_v2/probabilities
Output dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/unsw_nb15_v2


## 2. Load labels (defensive schema handling)

In [3]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)

# UNSW uses 'five_class_id_to_name' (vs NSL's 'multiclass_5')
mapping_5 = None
for key_candidate in ['multiclass_5', 'five_class_id_to_name', '5class', 'five_class', 'multiclass']:
    if key_candidate in class_info:
        mapping_5 = class_info[key_candidate]
        print(f"Using key: '{key_candidate}'")
        break

if mapping_5 is None:
    print(json.dumps(class_info, indent=2))
    raise KeyError('No 5-class mapping found.')

INT_TO_CATEGORY = {int(k): v for k, v in mapping_5.items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'\nCalibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print()

calib_counts_5 = Counter(y_calib_5)
calib_counts_b = Counter(y_calib_b)

print('Per-class calibration set sizes:')
print('  Binary:')
for c in [0, 1]:
    n = calib_counts_b[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {"Normal" if c == 0 else "Attack":8s}: n={n:>6,}  → {strat}')

print('  5-class:')
for c in range(5):
    n = calib_counts_5[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {CLASS_NAMES_5[c]:8s}: n={n:>6,}  → {strat}')

Using key: 'five_class_id_to_name'

Calibration set: 27,069, Test set: 63,461

Per-class calibration set sizes:
  Binary:
    Normal  : n=11,200  → isotonic
    Attack  : n=15,869  → isotonic
  5-class:
    Normal  : n=11,200  → isotonic
    DoS     : n= 2,453  → isotonic
    Probe   : n= 2,498  → isotonic
    R2L     : n=10,665  → isotonic
    U2R     : n=   253  → isotonic


## 3. Helper functions (identical to NSL/CIC hybrid)

In [4]:
def fit_calibrator(p_calib, y_indicator, n_class):
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        cal = LogisticRegression(C=1e10, solver='lbfgs')
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (probs >= lo) & (probs <= hi) if i == n_bins - 1 else (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)

def calibrate_model_per_class(p_calib_2d, y_calib, p_test_2d, y_test, n_classes):
    p_test_cal = np.zeros_like(p_test_2d)
    strategies, brier_pre, brier_post, ece_pre, ece_post = {}, {}, {}, {}, {}
    calib_counts = Counter(y_calib)

    for c in range(n_classes):
        y_calib_c = (y_calib == c).astype(int)
        y_test_c = (y_test == c).astype(int)
        n_c = calib_counts[c]
        cal, strat = fit_calibrator(p_calib_2d[:, c], y_calib_c, n_c)
        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_2d[:, c])
        strategies[c] = strat
        brier_pre[c] = float(brier_score_loss(y_test_c, p_test_2d[:, c]))
        brier_post[c] = float(brier_score_loss(y_test_c, p_test_cal[:, c]))
        ece_pre[c] = expected_calibration_error(p_test_2d[:, c], y_test_c, ECE_N_BINS)
        ece_post[c] = expected_calibration_error(p_test_cal[:, c], y_test_c, ECE_N_BINS)

    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    p_test_cal_renorm = p_test_cal / row_sums

    return {
        'p_test_cal': p_test_cal_renorm,
        'strategies': strategies,
        'brier_pre': brier_pre, 'brier_post': brier_post,
        'ece_pre': ece_pre, 'ece_post': ece_post,
    }

def calibrate_binary(p_calib_2d, y_calib, p_test_2d, y_test):
    p_calib_attack = p_calib_2d[:, 1]
    p_test_attack = p_test_2d[:, 1]
    calib_counts = Counter(y_calib)
    n_attack = calib_counts[1]
    cal, strat = fit_calibrator(p_calib_attack, y_calib, n_attack)
    p_test_cal_attack = apply_calibrator(cal, strat, p_test_attack)
    p_test_cal_attack = np.clip(p_test_cal_attack, 0, 1)
    p_test_cal_2d = np.column_stack([1 - p_test_cal_attack, p_test_cal_attack])
    return {
        'p_test_cal': p_test_cal_2d,
        'strategies': {1: strat},
        'brier_pre': {1: float(brier_score_loss(y_test, p_test_attack))},
        'brier_post': {1: float(brier_score_loss(y_test, p_test_cal_attack))},
        'ece_pre': {1: expected_calibration_error(p_test_attack, y_test, ECE_N_BINS)},
        'ece_post': {1: expected_calibration_error(p_test_cal_attack, y_test, ECE_N_BINS)},
    }

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 4. Calibrate all 9 models

In [5]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task})...')
    p_calib = np.load(PROBS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PROBS_DIR / f'{name}_test_proba.npy')

    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    if task == 'binary':
        result = calibrate_binary(p_calib, y_calib_b, p_test, y_test_b)
    else:
        result = calibrate_model_per_class(p_calib, y_calib_5, p_test, y_test_5, n_classes=5)

    np.save(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy', result['p_test_cal'])

    print(f'  Strategies: {result["strategies"]}')
    if task == '5class':
        for c in range(5):
            print(f'  {CLASS_NAMES_5[c]:8s}: Brier {result["brier_pre"][c]:.4f} → {result["brier_post"][c]:.4f}'
                  f'   ECE {result["ece_pre"][c]:.4f} → {result["ece_post"][c]:.4f}'
                  f'   [{result["strategies"][c]}]')
    else:
        print(f'  Attack  : Brier {result["brier_pre"][1]:.4f} → {result["brier_post"][1]:.4f}'
              f'   ECE {result["ece_pre"][1]:.4f} → {result["ece_post"][1]:.4f}'
              f'   [{result["strategies"][1]}]')

    calibration_summary[name] = {
        'task': task,
        'strategies': {int(k): v for k, v in result['strategies'].items()},
        'brier_pre':  {int(k): v for k, v in result['brier_pre'].items()},
        'brier_post': {int(k): v for k, v in result['brier_post'].items()},
        'ece_pre':  {int(k): v for k, v in result['ece_pre'].items()},
        'ece_post': {int(k): v for k, v in result['ece_post'].items()},
    }

with open(CALIB_OUT_DIR / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary: {CALIB_OUT_DIR}/calibration_summary.json')


Calibrating rf_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0951 → 0.1061   ECE 0.1018 → 0.1178   [isotonic]

Calibrating xgb_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0967 → 0.1087   ECE 0.1064 → 0.1225   [isotonic]

Calibrating dnn_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.0984 → 0.1160   ECE 0.1059 → 0.1302   [isotonic]

Calibrating rf_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'isotonic'}
  Normal  : Brier 0.1115 → 0.1098   ECE 0.1272 → 0.1202   [isotonic]
  DoS     : Brier 0.0502 → 0.0460   ECE 0.0368 → 0.0040   [isotonic]
  Probe   : Brier 0.0348 → 0.0281   ECE 0.0440 → 0.0188   [isotonic]
  R2L     : Brier 0.1517 → 0.1592   ECE 0.1030 → 0.1105   [isotonic]
  U2R     : Brier 0.0130 → 0.0059   ECE 0.0285 → 0.0067   [isotonic]

Calibrating xgb_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isoto

## 5. Macro summary

In [6]:
rows = []
for name, task in ALL_MODELS:
    s = calibration_summary[name]
    if task == '5class':
        brier_pre_macro = np.mean(list(s['brier_pre'].values()))
        brier_post_macro = np.mean(list(s['brier_post'].values()))
        ece_pre_macro = np.mean(list(s['ece_pre'].values()))
        ece_post_macro = np.mean(list(s['ece_post'].values()))
        platt_classes = [CLASS_NAMES_5[c] for c, st in s['strategies'].items() if st == 'platt']
        rows.append({
            'Model': name,
            'Task': '5-class',
            'Brier pre (macro)':  round(brier_pre_macro,  5),
            'Brier post (macro)': round(brier_post_macro, 5),
            'ECE pre (macro)':    round(ece_pre_macro,    5),
            'ECE post (macro)':   round(ece_post_macro,   5),
            'Platt for': ','.join(platt_classes) if platt_classes else '—',
        })
    else:
        rows.append({
            'Model': name,
            'Task': 'binary',
            'Brier pre (macro)':  round(s['brier_pre'][1],  5),
            'Brier post (macro)': round(s['brier_post'][1], 5),
            'ECE pre (macro)':    round(s['ece_pre'][1],    5),
            'ECE post (macro)':   round(s['ece_post'][1],   5),
            'Platt for': '—' if s['strategies'][1] == 'isotonic' else 'attack class',
        })

df = pd.DataFrame(rows)
print('\nUNSW v2 — Hybrid Platt/Isotonic Calibration Results')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLES_DIR / 'unsw_v2_calibration_summary.csv', index=False)
print(f'\nSaved: {TABLES_DIR}/unsw_v2_calibration_summary.csv')


UNSW v2 — Hybrid Platt/Isotonic Calibration Results
           Model    Task  Brier pre (macro)  Brier post (macro)  ECE pre (macro)  ECE post (macro) Platt for
    rf_binary_cw  binary            0.09506             0.10611          0.10181           0.11782         —
   xgb_binary_cw  binary            0.09669             0.10875          0.10637           0.12251         —
   dnn_binary_cw  binary            0.09840             0.11602          0.10587           0.13020         —
 rf_5class_smote 5-class            0.07224             0.06981          0.06790           0.05203         —
xgb_5class_smote 5-class            0.07125             0.06888          0.06258           0.05509         —
dnn_5class_smote 5-class            0.08687             0.07319          0.08761           0.05385         —
    rf_5class_cw 5-class            0.06965             0.06856          0.06199           0.05120         —
   xgb_5class_cw 5-class            0.07505             0.07055          0.

## 6. Per-class breakdown

In [7]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    s = calibration_summary[name]
    for c in range(5):
        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Strategy': s['strategies'][c],
            'Brier pre':  round(s['brier_pre'][c],  5),
            'Brier post': round(s['brier_post'][c], 5),
            'ECE pre':    round(s['ece_pre'][c],    5),
            'ECE post':   round(s['ece_post'][c],   5),
        })

df_perclass = pd.DataFrame(perclass_rows)
print('UNSW v2 — Per-class breakdown')
print('=' * 120)
print(df_perclass.to_string(index=False))
print('=' * 120)

df_perclass.to_csv(TABLES_DIR / 'unsw_v2_calibration_perclass.csv', index=False)
print(f'\nSaved: {TABLES_DIR}/unsw_v2_calibration_perclass.csv')

UNSW v2 — Per-class breakdown
           Model  Class  n_calib Strategy  Brier pre  Brier post  ECE pre  ECE post
 rf_5class_smote Normal    11200 isotonic    0.11150     0.10983  0.12724   0.12020
 rf_5class_smote    DoS     2453 isotonic    0.05019     0.04603  0.03682   0.00396
 rf_5class_smote  Probe     2498 isotonic    0.03481     0.02806  0.04400   0.01882
 rf_5class_smote    R2L    10665 isotonic    0.15167     0.15922  0.10295   0.11052
 rf_5class_smote    U2R      253 isotonic    0.01301     0.00592  0.02846   0.00666
xgb_5class_smote Normal    11200 isotonic    0.10682     0.10698  0.12399   0.12142
xgb_5class_smote    DoS     2453 isotonic    0.05211     0.04531  0.02817   0.00675
xgb_5class_smote  Probe     2498 isotonic    0.03517     0.02840  0.03657   0.02199
xgb_5class_smote    R2L    10665 isotonic    0.15350     0.15848  0.11211   0.11982
xgb_5class_smote    U2R      253 isotonic    0.00863     0.00522  0.01206   0.00549
dnn_5class_smote Normal    11200 isotonic    0

## 7. Argmax preservation check (inline, like CIC Dirichlet)

In [8]:
from sklearn.metrics import accuracy_score, f1_score

print('UNSW HYBRID — Argmax preservation check')
print('=' * 100)
print(f'{"Model":<22} {"% flipped":>10} {"Acc pre":>9} {"Acc post":>10} {"Δ acc":>9} '
      f'{"F1m pre":>9} {"F1m post":>10} {"Δ F1m":>9}')
print('-' * 100)

for name, task in ALL_MODELS:
    if task != '5class':
        continue
    p_pre = np.load(PROBS_DIR / f'{name}_test_proba.npy')
    p_post = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    n_flipped = (pred_pre != pred_post).sum()
    pct_flipped = 100 * n_flipped / len(pred_pre)

    acc_pre = accuracy_score(y_test_5, pred_pre)
    acc_post = accuracy_score(y_test_5, pred_post)
    f1_pre = f1_score(y_test_5, pred_pre, average='macro', zero_division=0)
    f1_post = f1_score(y_test_5, pred_post, average='macro', zero_division=0)

    print(f'{name:<22} {pct_flipped:>9.2f}% {acc_pre:>9.4f} {acc_post:>10.4f} {acc_post-acc_pre:>+9.4f} '
          f'{f1_pre:>9.4f} {f1_post:>10.4f} {f1_post-f1_pre:>+9.4f}')

UNSW HYBRID — Argmax preservation check
Model                   % flipped   Acc pre   Acc post     Δ acc   F1m pre   F1m post     Δ F1m
----------------------------------------------------------------------------------------------------
rf_5class_smote            18.74%    0.6872     0.7311   +0.0439    0.5505     0.5423   -0.0082
xgb_5class_smote           12.73%    0.7152     0.7423   +0.0271    0.5911     0.5742   -0.0169
dnn_5class_smote           23.65%    0.6348     0.7100   +0.0753    0.5020     0.5134   +0.0114
rf_5class_cw               17.38%    0.6961     0.7396   +0.0435    0.5686     0.5545   -0.0140
xgb_5class_cw              15.62%    0.7012     0.7304   +0.0292    0.5775     0.5738   -0.0037
dnn_5class_cw              25.42%    0.6017     0.6640   +0.0623    0.4774     0.4746   -0.0028


## 8. Sanity checks

In [9]:
print('Sanity checks:')
print('-' * 70)
for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5
    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()
    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

Sanity checks:
----------------------------------------------------------------------
  ✓ rf_binary_cw           shape=(63461, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_binary_cw          shape=(63461, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_binary_cw          shape=(63461, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_smote        shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_smote       shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_smote       shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_cw           shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_cw          shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_cw          shape=(63461, 5)  range_ok=True  sum_max_dev=0.00000  finite=True


## 9. Commit

In [10]:
os.chdir(REPO)
!git add notebooks/03_unsw_calibration_v2.ipynb
!git add calibrators/unsw_nb15_v2/
!git add results/tables/unsw_v2_calibration_summary.csv
!git add results/tables/unsw_v2_calibration_perclass.csv
!git status --short
!git commit -m 'Notebook 03-UNSW v2: hybrid Platt/isotonic calibration (all classes isotonic, U2R n=253)'
!git push origin main

Refresh index: 100% (259/259), done.
A  calibrators/unsw_nb15_v2/calibration_summary.json
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_cic_train_models_v2.ipynb
 M notebooks/02_nsl_rf_retrain_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
 M notebooks/03_cic_calibration_v2.ipynb
 M notebooks/03_nsl_calibration_v2.ipynb
AM notebooks/03_unsw_calibration_v2.ipynb
 M notebooks/03c_cic_calibration_dirichlet_v2.ipynb
 M notebooks/03c_nsl_calibration_dirichlet_v2.ipynb
 M notebooks/03c_unsw_calibration_dirichlet_v2.ipynb
A  results/tables/unsw_v2_calibration_perclass.csv
A  results/tables/unsw_v2_calibration_summary.csv
?? calibrators/nsl_kdd/
?? calibrators/unsw_nb15/
?? docs/v2_rebuild_progress_day1_day2_v7.md
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? notebooks/03b_nsl_calibration_diagnostic.ipynb
?? results/figures/diag_nsl_dnn5